# DRISHTI — YOLOv8 Fine-tuning Notebook
## Gridlock Hackathon 2.0 | Team: Chirantan, Jasmon, Chirag

Training a custom violation detection model on Indian traffic datasets.

**Runtime required: GPU (T4 or better).** Go to **Runtime → Change runtime type → GPU**

**Expected training time:** ~25–35 minutes on Colab free-tier T4 (50 epochs with early stopping).

**Only manual step:** Paste your Roboflow API key in Cell 4.

## Cell 2 — GPU Check
Verify Colab allocated a GPU before installing packages or training.

In [ ]:
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "⚠️ No GPU detected! Change runtime to GPU.")
print("✅ Done: GPU check")

## Cell 3 — Install Dependencies
Install Ultralytics YOLOv8 and Roboflow SDK, then run environment checks.

In [ ]:
!pip install ultralytics roboflow pyyaml matplotlib -q

import ultralytics
ultralytics.checks()
print("✅ Done: Install dependencies")

## Cell 4 — Dataset Download (Roboflow)
Downloads two public datasets:
- **Dataset A:** Helmet violation detection (Indian traffic)
- **Dataset B:** Vehicle detection (IDD-style)

Replace `YOUR_ROBOFLOW_API_KEY` below with your key from [Roboflow Settings](https://app.roboflow.com/settings/api).

In [ ]:
import glob
import matplotlib.pyplot as plt
from pathlib import Path
from roboflow import Roboflow

ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"

if not ROBOFLOW_API_KEY or ROBOFLOW_API_KEY == "YOUR_ROBOFLOW_API_KEY":
    raise ValueError("Please set your Roboflow API key in Cell 4")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Dataset A: Helmet detection — Indian traffic
project_helmet = rf.workspace("roboflow-universe-projects").project("helmet-detection-pn5d0")
dataset_helmet = project_helmet.version(5).download("yolov8")

# Dataset B: Vehicle detection — IDD-style
project_vehicle = rf.workspace("roboflow-universe-projects").project("vehicle-detection-3mmwj")
dataset_vehicle = project_vehicle.version(1).download("yolov8")

helmet_path = Path(dataset_helmet.location)
vehicle_path = Path(dataset_vehicle.location)

print("Helmet dataset:", helmet_path)
print("Vehicle dataset:", vehicle_path)


def show_sample_images(dataset_root, title, count=3):
    patterns = [
        str(dataset_root / "train" / "images" / "*"),
        str(dataset_root / "valid" / "images" / "*"),
    ]
    images = []
    for pattern in patterns:
        images.extend(glob.glob(pattern))
    images = images[:count]

    if not images:
        print(f"No images found for {title}")
        return

    fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4))
    if len(images) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, images):
        ax.imshow(plt.imread(img_path))
        ax.set_title(Path(img_path).name, fontsize=8)
        ax.axis("off")
    fig.suptitle(title)
    plt.show()


show_sample_images(helmet_path, "Helmet Dataset — sample images")
show_sample_images(vehicle_path, "Vehicle Dataset — sample images")
print("✅ Done: Dataset download")

## Cell 5 — Dataset Merge + Class Mapping

We merge both Roboflow exports into one YOLO dataset at `/content/drishti_dataset/`.

**Target classes (11):**
| ID | Class |
|----|-------|
| 0 | motorcycle |
| 1 | car |
| 2 | bus |
| 3 | truck |
| 4 | person |
| 5 | helmet_violation |
| 6 | triple_riding |
| 7 | stop_line_violation |
| 8 | red_light_violation |
| 9 | illegal_parking |
| 10 | wrong_side |

Source labels are remapped via alias matching (e.g. `no-helmet` → `helmet_violation`). Classes with no training data yet (triple riding, red light, etc.) remain in the schema for future labeling.

In [ ]:
import shutil
import yaml
from pathlib import Path

MERGED_ROOT = Path("/content/drishti_dataset")
TARGET_CLASSES = [
    "motorcycle", "car", "bus", "truck", "person",
    "helmet_violation", "triple_riding", "stop_line_violation",
    "red_light_violation", "illegal_parking", "wrong_side",
]

CLASS_ALIASES = {
    "motorcycle": 0, "motorbike": 0, "bike": 0, "two_wheeler": 0, "scooter": 0,
    "car": 1, "automobile": 1, "sedan": 1, "hatchback": 1,
    "bus": 2,
    "truck": 3, "lorry": 3, "pickup": 3,
    "person": 4, "pedestrian": 4, "rider": 4, "human": 4, "people": 4,
    "helmet_violation": 5, "no_helmet": 5, "without_helmet": 5, "nohelmet": 5,
    "helmetless": 5, "withouthelmet": 5, "no_helmet_rider": 5,
    "triple_riding": 6, "triple_ride": 6, "triple": 6,
    "stop_line_violation": 7, "stop_line": 7, "stopline": 7, "crossed_stop_line": 7,
    "red_light_violation": 8, "red_light": 8, "redlight": 8, "signal_jump": 8,
    "illegal_parking": 9, "parking_violation": 9, "wrong_parking": 9,
    "wrong_side": 10, "wrong_side_driving": 10, "wrong_way": 10,
}

SKIP_ALIASES = {"with_helmet", "helmet", "yes_helmet", "wearing_helmet", "number_plate", "license_plate"}


def normalize_name(name: str) -> str:
    return name.strip().lower().replace("-", "_").replace(" ", "_")


def load_yaml_names(dataset_root: Path) -> list:
    with open(dataset_root / "data.yaml", "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)
    return data.get("names", [])


def build_class_mapper(source_names: list) -> dict:
    mapping = {}
    for src_id, raw_name in enumerate(source_names):
        key = normalize_name(str(raw_name))
        if key in SKIP_ALIASES:
            continue
        if key in CLASS_ALIASES:
            mapping[src_id] = CLASS_ALIASES[key]
            continue
        for alias, target_id in CLASS_ALIASES.items():
            if alias in key or key in alias:
                mapping[src_id] = target_id
                break
    return mapping


def remap_label_file(src_label: Path, dst_label: Path, class_mapper: dict) -> bool:
    if not src_label.exists():
        return False

    remapped_lines = []
    for line in src_label.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        src_cls = int(float(parts[0]))
        if src_cls not in class_mapper:
            continue
        parts[0] = str(class_mapper[src_cls])
        remapped_lines.append(" ".join(parts))

    if not remapped_lines:
        return False

    dst_label.parent.mkdir(parents=True, exist_ok=True)
    dst_label.write_text("\n".join(remapped_lines) + "\n", encoding="utf-8")
    return True


def ingest_dataset(dataset_root: Path, prefix: str, class_mapper: dict) -> dict:
    counts = {"train": 0, "valid": 0, "test": 0}
    split_map = {"train": "train", "valid": "valid", "val": "valid", "test": "test"}

    for split_dir in dataset_root.iterdir():
        if not split_dir.is_dir():
            continue
        split_name = split_map.get(split_dir.name.lower())
        if not split_name:
            continue

        img_dir = split_dir / "images"
        lbl_dir = split_dir / "labels"
        if not img_dir.exists():
            continue

        out_img_dir = MERGED_ROOT / split_name / "images"
        out_lbl_dir = MERGED_ROOT / split_name / "labels"
        out_img_dir.mkdir(parents=True, exist_ok=True)
        out_lbl_dir.mkdir(parents=True, exist_ok=True)

        for img_path in sorted(img_dir.glob("*")):
            if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
                continue
            stem = f"{prefix}_{img_path.stem}"
            dst_img = out_img_dir / f"{stem}{img_path.suffix.lower()}"
            dst_lbl = out_lbl_dir / f"{stem}.txt"
            src_lbl = lbl_dir / f"{img_path.stem}.txt"

            if remap_label_file(src_lbl, dst_lbl, class_mapper):
                shutil.copy2(img_path, dst_img)
                counts[split_name] += 1

    return counts


if MERGED_ROOT.exists():
    shutil.rmtree(MERGED_ROOT)
MERGED_ROOT.mkdir(parents=True, exist_ok=True)

helmet_names = load_yaml_names(helmet_path)
vehicle_names = load_yaml_names(vehicle_path)
helmet_mapper = build_class_mapper(helmet_names)
vehicle_mapper = build_class_mapper(vehicle_names)

print("Helmet source classes:", helmet_names)
print("Helmet class map:", helmet_mapper)
print("Vehicle source classes:", vehicle_names)
print("Vehicle class map:", vehicle_mapper)

helmet_counts = ingest_dataset(helmet_path, "helmet", helmet_mapper)
vehicle_counts = ingest_dataset(vehicle_path, "vehicle", vehicle_mapper)

merged_yaml = {
    "path": str(MERGED_ROOT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(TARGET_CLASSES),
    "names": TARGET_CLASSES,
}

with open(MERGED_ROOT / "data.yaml", "w", encoding="utf-8") as f:
    yaml.dump(merged_yaml, f, default_flow_style=False, sort_keys=False)

train_count = len(list((MERGED_ROOT / "train" / "images").glob("*")))
valid_count = len(list((MERGED_ROOT / "valid" / "images").glob("*")))
test_count = len(list((MERGED_ROOT / "test" / "images").glob("*")))

print(f"Merged train images: {train_count}")
print(f"Merged valid images: {valid_count}")
print(f"Merged test images: {test_count}")
print(f"data.yaml written to: {MERGED_ROOT / 'data.yaml'}")
print("✅ Done: Dataset merge + class mapping")

## Cell 6 — Training Configuration
Fine-tune YOLOv8n on the merged DRISHTI dataset. Augmentation tuned for Indian traffic (lighting, weather, occlusion).

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # Start from nano pretrained weights

results = model.train(
    data=str(MERGED_ROOT / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    name="drishti_v1",
    project="/content/runs",
    device=0,
    patience=10,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.1,
    mosaic=1.0,
    mixup=0.1,
)

print("✅ Done: Training")

## Cell 7 — Training Results Visualization
Display training curves, confusion matrix, and final mAP metrics.

In [ ]:
from IPython.display import Image, display

run_dir = Path("/content/runs/drishti_v1")

for plot_name in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    plot_path = run_dir / plot_name
    if plot_path.exists():
        print(plot_name)
        display(Image(str(plot_path)))
    else:
        print(f"Missing plot: {plot_path}")

metrics = getattr(results, "results_dict", {}) or {}
map50 = metrics.get("metrics/mAP50(B)")
map50_95 = metrics.get("metrics/mAP50-95(B)")

if map50 is not None:
    print(f"mAP50: {map50:.4f}")
if map50_95 is not None:
    print(f"mAP50-95: {map50_95:.4f}")

print("✅ Done: Training results visualization")

## Cell 8 — Validation on Sample Images
Run inference on 5 validation images and display annotated outputs.

In [ ]:
val_images = sorted(glob.glob(str(MERGED_ROOT / "valid" / "images" / "*")))[:5]

if not val_images:
  val_images = sorted(glob.glob(str(MERGED_ROOT / "train" / "images" / "*")))[:5]

for i, img_path in enumerate(val_images):
    result = model(img_path)
    out_path = f"/content/val_output_{i}.jpg"
    result[0].save(filename=out_path)
    print(img_path)
    display(Image(out_path))

print("✅ Done: Validation on sample images")

## Cell 9 — Export Weights
Copy best checkpoint and download as `drishti_best.pt` for the FastAPI backend.

In [ ]:
import shutil
from google.colab import files

best_weights = Path("/content/runs/drishti_v1/weights/best.pt")
if not best_weights.exists():
    raise FileNotFoundError(f"Best weights not found at {best_weights}")

shutil.copy(best_weights, "/content/drishti_best.pt")
files.download("/content/drishti_best.pt")

print("✅ Download drishti_best.pt and place it at: drishti-backend/models/drishti.pt")
print("The backend auto-loads models/drishti.pt when present (no main.py change needed).")
print("✅ Done: Export weights")

## Cell 10 — Quick Performance Benchmark
Measure average inference latency on a sample validation image (GPU).

In [ ]:
import time

if not val_images:
    raise ValueError("No validation images available for benchmark.")

test_img = val_images[0]

# Warmup
model(test_img)

times = []
for _ in range(10):
    start = time.perf_counter()
    model(test_img)
    times.append((time.perf_counter() - start) * 1000)

avg_ms = sum(times) / len(times)
print(f"Average inference time: {avg_ms:.1f}ms")
print(f"Estimated FPS: {1000 / avg_ms:.1f}")
print("Note: Production deployment uses GPU — CPU inference will be slower")
print("✅ Done: Performance benchmark")

## Cell 11 — Final Summary

## Results Summary
Fill this in after training:
- Dataset size: __ training images, __ validation images
- Training epochs completed: __
- mAP50: __
- mAP50-95: __
- Best checkpoint: `drishti_best.pt`
- Next step: Rename downloaded weights to `drishti-backend/models/drishti.pt` and restart the backend

After placing weights, `/health` should report `"model": "drishti.pt"` and DEMO mock overlay is disabled automatically.